In [2]:
import sys, os

In [3]:
# notebook parameters, will be replaced by script arguments
application_name = "My Application"
s3a_access_key = os.environ.get("s3a_access_key")
s3a_secret_key = os.environ.get("s3a_secret_key")
input_file_1 = "s3a://uga-data-lake/sensor-streamer-test2/android-sensor-*/*/*/*"
output_file_1 = "s3a://uga-data-lake/sensor-streamer-test2/analytics"
output_file_2 = "s3a://uga-data-lake/sensor-streamer-test2/checkpoint"

# The demo will use the app [Android Sensor Cloud Streamer](https://androidsensor.ai.studio/)

In [4]:
import pyspark
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
import pyspark.sql.types as T
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, BooleanType, DoubleType, LongType
import pyspark.sql.functions as F
pyspark.__version__

'4.2.0'

In [5]:
# test if the script is running as a standalone program or a notebook
if __name__ == '__main__' and '__file__' in globals():
    print("Running as a script")
    application_name = sys.argv[1]
    s3a_access_key = sys.argv[2]
    s3a_secret_key = sys.argv[3]
    input_file_1 = sys.argv[4]
    output_file_1 = sys.argv[5]
    output_file_2 = sys.argv[6]
    interactive = False
else:
    interactive = True
    print("Running as a notebook")
    pv = !{sys.executable} --version
    print("Python version ", pv)
    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
    print("PySpark version ", pyspark.__version__)
    jv = !java --version
    print("Java version ", jv)

Running as a notebook
Python version  ['Python 3.12.15']
PySpark version  4.2.0
Java version  ['openjdk 17.0.19 2026-04-21', 'OpenJDK Runtime Environment (build 17.0.19+10-1-deb12u2-Debian)', 'OpenJDK 64-Bit Server VM (build 17.0.19+10-1-deb12u2-Debian, mixed mode, sharing)']


In [6]:
def show(df):
    if interactive:
        df.show()

In [7]:
spark = (SparkSession.builder.appName(application_name)
         .config('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.5.0')
         .config('spark.hadoop.fs.s3a.access.key', s3a_access_key)
         .config("spark.hadoop.fs.s3a.secret.key", s3a_secret_key)
         .config("spark.hadoop.fs.s3a.endpoint", "s3.us-east-2.amazonaws.com")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ba66cb18-2e00-4dc0-8772-a8b0a846508e;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.5.0 in central
	found software.amazon.awssdk#bundle;2.35.4 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.3.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
:: resolution report :: resolve 128ms :: artifacts dl 5ms
	:: modules in use:
	org.apache.hadoop#hadoop-aws;3.5.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;2.2.5.Final from central in [default]
	software.amazon.awssdk#bundle;2.35.4 from central in [default]
	software.amazon.s3.analyt

In [32]:
sensor_payload_schema = StructType([StructField('activeSensors', ArrayType(StringType(), True), True), StructField('deviceId', StringType(), True), StructField('deviceMetrics', StructType([StructField('online', BooleanType(), True), StructField('userAgent', StringType(), True)]), True), StructField('epochMs', LongType(), True), StructField('readings', StructType([StructField('accelerometer', StructType([StructField('current', StructType([StructField('magnitude', DoubleType(), True), StructField('x', DoubleType(), True), StructField('y', DoubleType(), True), StructField('z', DoubleType(), True)]), True), StructField('recentSamples', ArrayType(StructType([StructField('t', LongType(), True), StructField('x', DoubleType(), True), StructField('y', DoubleType(), True), StructField('z', DoubleType(), True)]), True), True), StructField('stats', StructType([StructField('magnitudeMax', DoubleType(), True), StructField('magnitudeMin', DoubleType(), True), StructField('magnitudeVariance', DoubleType(), True), StructField('meanX', DoubleType(), True), StructField('meanY', DoubleType(), True), StructField('meanZ', DoubleType(), True), StructField('potentialAnomaly', BooleanType(), True), StructField('sampleCount', LongType(), True)]), True)]), True), StructField('gyroscope', StructType([StructField('current', StructType([StructField('alpha', DoubleType(), True), StructField('beta', DoubleType(), True), StructField('gamma', DoubleType(), True), StructField('rotationMagnitude', DoubleType(), True)]), True), StructField('recentSamples', ArrayType(StructType([StructField('a', DoubleType(), True), StructField('b', DoubleType(), True), StructField('g', DoubleType(), True), StructField('t', LongType(), True)]), True), True), StructField('stats', StructType([StructField('meanAlpha', DoubleType(), True), StructField('meanBeta', DoubleType(), True), StructField('meanGamma', DoubleType(), True), StructField('peakRotationRate', DoubleType(), True), StructField('sampleCount', LongType(), True)]), True)]), True), StructField('location', StructType([StructField('accuracy', DoubleType(), True), StructField('altitude', DoubleType(), True), StructField('latitude', DoubleType(), True), StructField('longitude', DoubleType(), True), StructField('speed', DoubleType(), True)]), True)]), True), StructField('timestamp', StringType(), True), StructField('updatePeriodSeconds', LongType(), True)])

In [33]:
# raw_df = (spark.read
#     .option("multiLine", True).json(input_file_1)
# )
# raw_df.printSchema()

In [36]:
raw_stream_df = (spark.readStream
    .schema(sensor_payload_schema)
    .option("multiLine", True).json(input_file_1)
)
raw_stream_df.isStreaming
raw_stream_df.printSchema()

root
 |-- activeSensors: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- deviceId: string (nullable = true)
 |-- deviceMetrics: struct (nullable = true)
 |    |-- online: boolean (nullable = true)
 |    |-- userAgent: string (nullable = true)
 |-- epochMs: long (nullable = true)
 |-- readings: struct (nullable = true)
 |    |-- accelerometer: struct (nullable = true)
 |    |    |-- current: struct (nullable = true)
 |    |    |    |-- magnitude: double (nullable = true)
 |    |    |    |-- x: double (nullable = true)
 |    |    |    |-- y: double (nullable = true)
 |    |    |    |-- z: double (nullable = true)
 |    |    |-- recentSamples: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- t: long (nullable = true)
 |    |    |    |    |-- x: double (nullable = true)
 |    |    |    |    |-- y: double (nullable = true)
 |    |    |    |    |-- z: double (nullable = true)
 |    |    |-- stats: struct (

In [35]:
if interactive:
    q1=raw_stream_df.writeStream.format("console").outputMode("update").option("truncate", "false").queryName("readings").start()
    q1.awaitTermination(10)
    q1.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------------------------------------+----------------------+-----------------------------------------------------------------------------------------------------------------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Fall and impact detection

In [43]:
# Convert epoch milliseconds to event timestamp with watermark
flattened_stream = (raw_stream_df
    .withColumn("event_time", F.to_timestamp(F.from_unixtime(F.col("epochMs") / 1000)))
    .withWatermark("event_time", "20 seconds")
    .select(
        F.col("deviceId"),
        F.col("event_time"),
        F.col("readings.accelerometer.stats.magnitudeMin").alias("accel_min_magnitude"),
        F.col("readings.accelerometer.stats.magnitudeMax").alias("accel_max_magnitude"),
        F.col("readings.accelerometer.stats.magnitudeVariance").alias("accel_variance"),
        F.col("readings.gyroscope.stats.peakRotationRate").alias("gyro_peak_rotation"),
        F.col("readings.location.altitude").alias("altitude"),
        F.col("readings.location.latitude").alias("latitude"),
        F.col("readings.location.longitude").alias("longitude")
    )
)

windowed_aggregates = (flattened_stream
    .groupBy(
        F.col("deviceId"),
        F.window(F.col("event_time"), "10 seconds", "10 seconds")
    )
    .agg(
        F.min("accel_min_magnitude").alias("window_min_accel"),
        F.max("accel_max_magnitude").alias("window_max_accel"),
        F.avg("accel_variance").alias("window_avg_variance"),
        F.avg("altitude").alias("window_avg_altitude"),
        F.avg("latitude").alias("window_avg_latitude"),
        F.avg("longitude").alias("window_avg_longitude"),
        F.max("gyro_peak_rotation").alias("window_max_rotation"),
        F.count("*").alias("batch_count")
    )
)

query_impact = (windowed_aggregates
    .filter(F.col("window_max_accel") >= 24.0)
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("deviceId"),
        F.col("window_avg_altitude"),
        F.col("window_avg_latitude"),
        F.col("window_avg_longitude"),
        F.col("window_max_rotation"),
        F.lit("HIGH_IMPACT_DETECTED").alias("anomaly_type"),
        F.round(F.col("window_max_accel"), 2).alias("peak_impact_ms2"),
        F.when(F.col("window_max_accel") >= 40.0, "SEVERE_COLLISION")
         .when(F.col("window_max_accel") >= 30.0, "HARD_IMPACT")
         .otherwise("MODERATE_IMPACT").alias("impact_class")
    )
)

In [42]:
if interactive:
    query = (query_impact.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", "false")
    .queryName("query_impact_detection")
    .start() )
    query.awaitTermination(10)
    query.stop()
else:
    query = (query_impact.writeStream
    .format("parquet")
    .option("path", output_file_1)
    .option("checkpointLocation", output_file_2)    
    .outputMode("append")
    .queryName("query_impact_detection1")
    .start() )    
    query.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------------+-------------------+----------------------+-------------------+--------------------+-------------------+--------------------+---------------+------------+
|window_start       |window_end         |deviceId              |window_avg_altitude|window_avg_longitude|window_max_rotation|anomaly_type        |peak_impact_ms2|impact_class|
+-------------------+-------------------+----------------------+-------------------+--------------------+-------------------+--------------------+---------------+------------+
|2026-11-07 00:33:00|2026-11-07 00:33:10|android-sensor-hanisaf|223.9              |-83.555693          |591.521            |HIGH_IMPACT_DETECTED|37.71          |HARD_IMPACT |
+-------------------+-------------------+----------------------+-------------------+--------------------+-------------------+--------------------+---------------+------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------+----------+--------+-------------------+--------------------+-------------------+------------+---------------+------------+
|window_start|window_end|deviceId|window_avg_altitude|window_avg_longitude|window_max_rotation|anomaly_type|peak_impact_ms2|impact_class|
+------------+----------+--------+-------------------+--------------------+-------------------+------------+---------------+------------+
+------------+----------+--------+-------------------+--------------------+-------------------+------------+---------------+------------+



In [8]:
if interactive:
    df = spark.read.parquet(output_file_1)
    df.show(truncate=False, vertical=True)
    df.printSchema()
    df.toPandas().to_csv("out/impacts.csv")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


-RECORD 0--------------------------------------
 window_start         | 2026-11-07 00:46:20    
 window_end           | 2026-11-07 00:46:30    
 deviceId             | android-sensor-hanisaf 
 window_avg_altitude  | 223.9                  
 window_avg_latitude  | 33.918371              
 window_avg_longitude | -83.555694             
 window_max_rotation  | 952.096                
 anomaly_type         | HIGH_IMPACT_DETECTED   
 peak_impact_ms2      | 57.64                  
 impact_class         | SEVERE_COLLISION       
-RECORD 1--------------------------------------
 window_start         | 2026-11-07 00:45:50    
 window_end           | 2026-11-07 00:46:00    
 deviceId             | android-sensor-hanisaf 
 window_avg_altitude  | 223.9                  
 window_avg_latitude  | 33.918375              
 window_avg_longitude | -83.555704             
 window_max_rotation  | 325.24                 
 anomaly_type         | HIGH_IMPACT_DETECTED   
 peak_impact_ms2      | 42.08           

/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/usr/local/lib/python3.12/site-packages/pyspark/sql/pandas/types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/usr/local/lib/python3.12/site-pack

NameError: name 'df' is not defined